# Per-unit attested twin of an SO-101

A calibrated `Robot` models a *design*; a `Twin` models a *specific unit* — the calibrated body **plus** the off-body physics that unit exhibits (per-servo heat), plus its identity (serial + provenance).

This notebook builds an SO-101 twin, reads its **attestation handle**, and forecasts the **servo temperature** under a sustained hold — the off-body state the body model alone never sees, and the signal the predictive self-check will derate/veto on.

In [1]:
import jax.numpy as jnp
from jaxterity.twin import so101_twin

twin = so101_twin("SO101-0001")
print("serial:", twin.serial, "| servos (DOFs):", twin.n_servos, "| battery:", twin.has_battery)

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'


serial: SO101-0001 | servos (DOFs): 6 | battery: False


## Attestation handle

The per-unit content hash (ADR-008), extended off-body. It moves when the robot model/params, the off-body params, the serial, or the provenance change. Two units with identical parameters but different serials get **distinct** handles.

In [2]:
h = twin.attestation_handle
other = twin.with_serial("SO101-0002")
print("handle:        ", h)
print("sibling handle:", other.attestation_handle)
print("distinct:      ", h != other.attestation_handle)

handle:         ed92592f5c09d9eee49d346027ec439bce403c081690bb30e448d747cfc1ad60
sibling handle: c917083e397ad5697a91b58a4f73f5619f85f99d5c5dc2969370e90f015170af
distinct:       True


## Predicted servo temperature under a sustained hold

Roll the twin forward under a constant holding torque on the fast `functional_dynamics` path. The body barely moves, but the servos heat up — the prediction the self-check consumes.

In [3]:
nq = twin.robot.build_system()._nq
nv = twin.n_servos
dt, seconds, hold = 0.02, 30.0, 0.6
n_steps = int(seconds / dt)
q0, qd0 = jnp.zeros(nq), jnp.zeros(nv)
T0 = jnp.full(nv, twin.T_amb)
tau = jnp.tile(jnp.full(nv, hold), (n_steps, 1))
out = twin.rollout(q0, qd0, T0, tau, dt)
T_final = float(out.temperature[-1].max())
print(f"ambient T0       = {twin.T_amb:.2f} K")
print(f"predicted Tfinal = {T_final:.2f} K")
print(f"rise             = {T_final - twin.T_amb:.2f} K  (off-body; unseen by the body model)")

ambient T0       = 298.15 K
predicted Tfinal = 301.55 K
rise             = 3.40 K  (off-body; unseen by the body model)


The rise is the catch: a body-only model says "torque available, proceed"; the twin predicts the servo crossing its thermal ceiling first. The safety monitor turns this forecast into a derate/veto decision.